In [ ]:
from guppylang.std.builtins import array, qubit
from guppylang import guppy
from guppylang.std.quantum import measure_array, x, h, measure, cx, cz

from hugr.hugr.render import RenderConfig

n = guppy.nat_var("n")

@guppy.unitary
class foo:

    @guppy
    def __call__(q1: array[qubit, n]) -> None:
        h(q1[0])

    @guppy
    def call_daggered(q1: array[qubit, n]) -> None:
        x(q1[0])

    @guppy
    def call_controlled(q1: array[qubit, n], _controls: array[qubit, n]) -> None:
        cz(_controls[0], q1[0])

    @guppy
    def call_ctrl_daggered(q1: array[qubit, n], _controls: array[qubit, n]) -> None:
        cx(_controls[0], q1[0])
        

@guppy
def main() -> None:
    q1 = array(qubit())
    foo(q1)
    measure_array(q1)

main.check()
main.compile().modules[0].render_dot(RenderConfig(max_node_label_length=50, max_metadata_length=None)).view()


In [1]:
from collections.abc import Callable

from guppylang.decorator import guppy
from guppylang.std.array import array

from guppylang.std.builtins import (
    Controllable,
    Daggerable,
    Unitary,
    control,
    dagger,
    owned,
    power,
)
from guppylang.std.num import nat
from guppylang.std.quantum import (
    angle,
    cx,
    discard,
    h,
    measure_array,
    qubit,
    rx,
    discard_array,
)
from guppylang_internals.metadata.common import (
    CONTROLLED_KEY,
    CTRL_DAGGERED_KEY,
    DAGGERED_KEY,
)
from hugr.ops import FuncDefn

def test_custom_modifier(validate):

    n = guppy.nat_var("n")
    @guppy.unitary
    class foo:
        
        @guppy
        def __call__(q1: array[qubit, n]) -> None:
            h(q1[0])

        @guppy
        def call_daggered(q1: array[qubit, n]) -> None:
            h(q1[0])

        @guppy
        def call_controlled(q1: array[qubit, n], _controls: array[qubit, n]) -> None:
            h(_controls[0])

        @guppy
        def call_ctrl_daggered(q1: array[qubit, n], _controls: array[qubit, n]) -> None:
            h(_controls[0])

    @guppy
    def main() -> None:
        q1 = array(qubit())
        foo(q1)
        measure_array(q1)

    package = main.compile()
    hugr_module = package.modules[0]
    for _, data in hugr_module.nodes():
        if (
            isinstance(data.op, FuncDefn)
            and data.op.f_name == "__main__.foo.__call__$1"
        ):
            assert data.metadata[DAGGERED_KEY] == "__main__.foo.call_daggered$1"
            assert data.metadata[CONTROLLED_KEY] == "__main__.foo.call_controlled$1"
            assert (
                data.metadata[CTRL_DAGGERED_KEY] == "__main__.foo.call_ctrl_daggered$1"
            )

test_custom_modifier(None)

Error: Variable not defined (at <In[1]>:40:38)
   | 
38 | 
39 |         @guppy
40 |         def __call__(q1: array[qubit, n]) -> None:
   |                                       ^ `n` is not defined

Guppy compilation failed due to 1 previous error
